<a href="https://www.kaggle.com/code/vidushigupta1/paper-opi-lt-edi-acl2022-detecting-signs-of-depr?scriptVersionId=331135844" target="_blank"><img align="left" alt="Kaggle" title="Open in Kaggle" src="https://kaggle.com/static/images/open-in-kaggle.svg"></a>

In [33]:
!pip install -q -U transformers simpletransformers

In [34]:
!git clone https://github.com/rafalposwiata/depression-detection-lt-edi-2022.git
import pandas as pd

train_path = "depression-detection-lt-edi-2022/data/preprocessed_dataset"
!ls {train_path}

fatal: destination path 'depression-detection-lt-edi-2022' already exists and is not an empty directory.
dev.csv  test.csv  train.csv


In [35]:
train_df = pd.read_csv(f"{train_path}/train.csv")
dev_df = pd.read_csv(f"{train_path}/dev.csv")
test_df = pd.read_csv(f"{train_path}/test.csv")

print(train_df.shape)
print(train_df.columns.tolist())
train_df.head()

(6006, 3)
['pid', 'text', 'labels']


,pid,text,labels
0,train_pid_7991,At this point just genuinely curious what sort...,0
1,train_pid_7992,I have literally tried everything.... : I'm st...,0
2,train_pid_7995,I'm really struggling : So I don't know how to...,0
3,train_pid_7996,My meds aren’t making my depression any better...,0
4,train_pid_7997,"Hi I'm unwell : I'm 21 now, ""vice ridden"", and...",0


In [36]:
print(train_df['labels'].value_counts().sort_index())
print(dev_df['labels'].value_counts().sort_index())

labels
0     650
1    3101
2    2255
Name: count, dtype: int64
labels
0     90
1    510
2    400
Name: count, dtype: int64


In [37]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import torch

tok = AutoTokenizer.from_pretrained("rafalposwiata/roberta-large-depression")
mdl = AutoModelForSequenceClassification.from_pretrained("rafalposwiata/roberta-large-depression").to("cuda").eval()

Loading weights:   0%|          | 0/393 [00:00<?, ?it/s]

In [38]:
def get_predictions(model, tokenizer, texts, batch_size=16, max_length=256):
    model.eval()
    all_preds = []
    for i in range(0, len(texts), batch_size):
        batch = texts[i:i+batch_size]
        inputs = tokenizer(batch, padding=True, truncation=True, max_length=max_length, return_tensors="pt").to("cuda")
        with torch.no_grad():
            logits = model(**inputs).logits
        all_preds.extend(torch.argmax(logits, dim=1).cpu().tolist())
    return all_preds

preds = get_predictions(mdl, tok, dev_df["text"].tolist())

In [39]:
from sklearn.metrics import classification_report

print("Authors' pretrained model:")
print(classification_report(dev_df["labels"], preds, target_names=["not depressed", "moderate", "severe"]))

Authors' pretrained model:
               precision    recall  f1-score   support

not depressed       0.54      0.41      0.47        90
     moderate       0.70      0.76      0.73       510
       severe       0.72      0.68      0.70       400

     accuracy                           0.70      1000
    macro avg       0.65      0.62      0.63      1000
 weighted avg       0.69      0.70      0.69      1000



In [40]:
from simpletransformers.classification import ClassificationArgs

model_args = ClassificationArgs(
    num_train_epochs=3,
    train_batch_size=8,
    eval_batch_size=8,
    learning_rate=1e-5,
    overwrite_output_dir=True,
    output_dir="outputs_deproberta/",
    no_cache=True,
    no_save=False
)

In [41]:
import torch
from torch.utils.data import Dataset, DataLoader

class DepressionDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_length=256):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        enc = self.tokenizer(
            self.texts[idx],
            truncation=True,
            max_length=self.max_length,
            padding="max_length",
            return_tensors="pt"
        )
        item = {k: v.squeeze(0) for k, v in enc.items()}
        item["labels"] = torch.tensor(self.labels[idx], dtype=torch.long)
        return item

train_dataset = DepressionDataset(train_df["text"].tolist(), train_df["labels"].tolist(), tok)
train_loader = DataLoader(train_dataset, batch_size=8, shuffle=True)

In [42]:
from transformers import AutoModelForSequenceClassification, get_linear_schedule_with_warmup
from torch.optim import AdamW
from tqdm import tqdm

train_model = AutoModelForSequenceClassification.from_pretrained(
    "rafalposwiata/deproberta-large-v1", num_labels=3
).to("cuda")

optimizer = AdamW(train_model.parameters(), lr=1e-5)
num_epochs = 3
total_steps = len(train_loader) * num_epochs
scheduler = get_linear_schedule_with_warmup(optimizer, num_warmup_steps=0, num_training_steps=total_steps)

train_model.train()
for epoch in range(num_epochs):
    total_loss = 0
    for batch in tqdm(train_loader, desc=f"Epoch {epoch+1}/{num_epochs}"):
        batch = {k: v.to("cuda") for k, v in batch.items()}
        optimizer.zero_grad()
        outputs = train_model(**batch)
        loss = outputs.loss
        loss.backward()
        optimizer.step()
        scheduler.step()
        total_loss += loss.item()
    print(f"Epoch {epoch+1} avg loss: {total_loss / len(train_loader):.4f}")

Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

[transformers] RobertaForSequenceClassification LOAD REPORT from: rafalposwiata/deproberta-large-v1
Key                        | Status     | 
---------------------------+------------+-
lm_head.layer_norm.bias    | UNEXPECTED | 
lm_head.dense.bias         | UNEXPECTED | 
lm_head.bias               | UNEXPECTED | 
lm_head.dense.weight       | UNEXPECTED | 
lm_head.layer_norm.weight  | UNEXPECTED | 
classifier.dense.weight    | MISSING    | 
classifier.out_proj.bias   | MISSING    | 
classifier.dense.bias      | MISSING    | 
classifier.out_proj.weight | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
Epoch 1/3: 100%|██████████| 751/751 [15:31<00:00,  1.24s/it]


Epoch 1 avg loss: 0.8209


Epoch 2/3: 100%|██████████| 751/751 [15:30<00:00,  1.24s/it]


Epoch 2 avg loss: 0.6656


Epoch 3/3: 100%|██████████| 751/751 [15:31<00:00,  1.24s/it]

Epoch 3 avg loss: 0.5310


In [43]:
train_model.eval()
preds2 = get_predictions(train_model, tok, dev_df["text"].tolist())

print("Your fine-tuned-from-DepRoBERTa model:")
print(classification_report(dev_df["labels"], preds2, target_names=["not depressed", "moderate", "severe"]))

Your fine-tuned-from-DepRoBERTa model:
               precision    recall  f1-score   support

not depressed       0.49      0.53      0.51        90
     moderate       0.63      0.77      0.69       510
       severe       0.68      0.48      0.56       400

     accuracy                           0.63      1000
    macro avg       0.60      0.59      0.59      1000
 weighted avg       0.64      0.63      0.62      1000



In [44]:
print("Authors' pretrained model:")
print(classification_report(dev_df["labels"], preds, target_names=["not depressed", "moderate", "severe"]))

print("\nYour fine-tuned-from-DepRoBERTa model:")
print(classification_report(dev_df["labels"], preds2, target_names=["not depressed", "moderate", "severe"]))

Authors' pretrained model:
               precision    recall  f1-score   support

not depressed       0.54      0.41      0.47        90
     moderate       0.70      0.76      0.73       510
       severe       0.72      0.68      0.70       400

     accuracy                           0.70      1000
    macro avg       0.65      0.62      0.63      1000
 weighted avg       0.69      0.70      0.69      1000


Your fine-tuned-from-DepRoBERTa model:
               precision    recall  f1-score   support

not depressed       0.49      0.53      0.51        90
     moderate       0.63      0.77      0.69       510
       severe       0.68      0.48      0.56       400

     accuracy                           0.63      1000
    macro avg       0.60      0.59      0.59      1000
 weighted avg       0.64      0.63      0.62      1000

